# Title

In [2]:
%config InteractiveShell.ast_node_interactivity='last_expr_or_assign'  # always print last expr.
%config InlineBackend.figure_format = 'svg'
%load_ext autoreload
%autoreload 2
%matplotlib inline

import logging
logging.basicConfig(level=logging.INFO)

In [3]:
import numpy as np
import gc
import logging
import os
from collections.abc import Callable
from copy import deepcopy
from dataclasses import dataclass
from functools import wraps, update_wrapper
from inspect import Parameter, signature
from time import perf_counter_ns
from types import MethodType
from typing import Any, Optional, Union, overload

np.set_printoptions(precision=4, floatmode='fixed', suppress=True)
rng = np.random.default_rng()

Generator(PCG64) at 0x7FA0935279E0

In [4]:
import logging
from collections import OrderedDict
from typing import Any, TypeVar
import torch
from torch import Tensor, jit, nn

from tsdm.models.generic.dense import ReverseDense
from tsdm.util import deep_dict_update, initialize_from_config
from tsdm.util.decorators import trace

INFO:numexpr.utils:Note: NumExpr detected 24 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 8.
INFO:numexpr.utils:NumExpr defaulting to 8 threads.
INFO:tsdm.config._config:Available Models: {'TFT', 'NODE', 'Latent-ODE', 'N-BEATS', 'ODE-RNN', 'TPA', 'SET-TS', 'mTAN', 'IP-Net', 'GRU-ODE-Bayes', 'M-RNN', 'NCDE', 'BRITS', 'Informer'}
INFO:tsdm.config._config:Available Datasets: {'MuJoCo', 'Electricity', 'M4', 'Household Consumption', 'UWAVE', 'Traffic', 'M5', 'tourism1', 'Physionet 2012', 'Human Activity', 'tourism2', 'Air Quality', 'M3', 'USHCN', 'Character Trajectories', 'Air Quality Multi-Site', 'Physionet 2019'}
INFO:tsdm.config._config:Initializing folder structure
INFO:tsdm.config._config:creating folder /home/rscholz/.tsdm/datasets
INFO:tsdm.config._config:creating folder /home/rscholz/.tsdm/models
INFO:tsdm.config._config:creating folder /home/rscholz/.tsdm/logs
INFO:tsdm.config._config:creating folder /home/rscholz/.tsdm/rawdata
INFO:tsdm.config._config:Create

In [5]:
nnModuleType = TypeVar("nnModuleType", bound=nn.Module)

~nnModuleType

In [6]:
wraps?

Signature:
wraps(
    wrapped,
    assigned=('__module__', '__name__', '__qualname__', '__doc__', '__annotations__'),
    updated=('__dict__',),
)
Docstring:
Decorator factory to apply update_wrapper() to a wrapper function

Returns a decorator that invokes update_wrapper() with the decorated
function as the wrapper argument and the arguments to wraps() as the
remaining arguments. Default arguments are as for update_wrapper().
This is a convenience function to simplify applying partial() to
update_wrapper().
File:      ~/miniconda3/envs/kiwi/lib/python3.9/functools.py
Type:      function


In [8]:
type(f"AutoJIT@{cls.__name__}")

Init signature: type(self, /, *args, **kwargs)
Docstring:     
type(object) -> the object's type
type(name, bases, dict, **kwds) -> a new type
Type:           type
Subclasses:     ABCMeta, EnumMeta, NamedTupleMeta, _TypedDictMeta, _ABC, MetaHasDescriptors, PyCStructType, UnionType, PyCPointerType, PyCArrayType, ...


$${\displaystyle \|A\|_{p,q}=\left(\frac{1}{n}\sum _{j=1}^{n}\left(\frac{1}{m}\sum _{i=1}^{m}|a_{ij}|^{p}\right)^{\frac {q}{p}}\right)^{\frac {1}{q}}.}$$

In [ ]:
xx



In [15]:
def autojit(base_class: type[nnModuleType], /, *, inherit: bool = False) -> type[nnModuleType]:
    assert issubclass(base_class, nn.Module)  
    
    @wraps(base_class, updated=())
    class WrappedClass(base_class):  # type: ignore  # pylint: disable=too-few-public-methods
        r"""A simple Wrapper."""
        @trace
        def __new__(cls, *args: Any, **kwargs: Any) -> nnModuleType:  # type: ignore[misc]      
            print(f"{cls=}, {args=}, {kwargs=}")
            instance: nnModuleType = super().__new__(cls)
            instance.__init__(*args, **kwargs)
            scripted: nnModuleType = jit.script(instance)
            # If __new__() does not return an instance of cls, then the new instance’s __init__() method will not be invoked!            
            return scripted
    assert issubclass(WrappedClass, base_class)
    return WrappedClass

In [16]:

@autojit
class Series(nn.ModuleList):
    """A ResNet model."""

    @trace
    def __init__(self, *modules: nn.Module) -> None:
        print("__INIT__ CALLED")
        super().__init__(modules)

    @jit.export
    def forward(self, x: Tensor) -> Tensor:
        r"""Forward pass.

        Parameters
        ----------
        x: Tensor

        Returns
        -------
        Tensor
        """
        for block in self:
            x = block(x)
        return x

    
# @autojit
class ResNet(Series):
    
    @trace
    def __init__(self, *args, **kwargs) -> None: 
        super().__init__(*args, **kwargs)
    
    @jit.export
    def forward(self, x: Tensor) -> Tensor:
        r"""Forward pass.

        Parameters
        ----------
        x: Tensor

        Returns
        -------
        Tensor
        """
        for block in self:
            x = x + block(x)
        return x

In [17]:
blocks = [
    nn.Linear(4,4),
    nn.ReLU(),
    nn.Linear(4,4),
]
x = torch.randn(7,4)
model = Series(*blocks)

INFO:trace: autojit.<locals>.WrappedClass.__new__: ENTERING	
args=tuple(
    <class '__main__.Series'>,
    Linear(in_features=4, out_features=4, bias=True),
    ReLU(),
    Linear(in_features=4, out_features=4, bias=True),
)	
kwargs=dict(
)
INFO:trace: Series.__init__: ENTERING	
args=tuple(
    <class '__main__.Series'>,
    Linear(in_features=4, out_features=4, bias=True),
    ReLU(),
    Linear(in_features=4, out_features=4, bias=True),
)	
kwargs=dict(
)
INFO:trace: Series.__init__: SUCCESS with result=None
INFO:trace: Series.__init__: EXITING
INFO:trace: autojit.<locals>.WrappedClass.__new__: SUCCESS with result=Series(
  (0): Linear(in_features=4, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=4, bias=True)
)
INFO:trace: autojit.<locals>.WrappedClass.__new__: EXITING
INFO:trace: Series.__init__: ENTERING	
args=tuple(
    Series(
      (0): Linear(in_features=4, out_features=4, bias=True)
      (1): ReLU()
      (2): Linear(in_features=4, out_fea

cls=<class '__main__.Series'>, args=(Linear(in_features=4, out_features=4, bias=True), ReLU(), Linear(in_features=4, out_features=4, bias=True)), kwargs={}
__INIT__ CALLED
__INIT__ CALLED


Series(
  (0): Linear(in_features=4, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=4, bias=True)
)

In [18]:
y = model(x)
torch.linalg.norm(y).backward()

In [19]:
model = ResNet(*blocks)

INFO:trace: autojit.<locals>.WrappedClass.__new__: ENTERING	
args=tuple(
    <class '__main__.ResNet'>,
    Linear(in_features=4, out_features=4, bias=True),
    ReLU(),
    Linear(in_features=4, out_features=4, bias=True),
)	
kwargs=dict(
)
INFO:trace: ResNet.__init__: ENTERING	
args=tuple(
    <class '__main__.ResNet'>,
    Linear(in_features=4, out_features=4, bias=True),
    ReLU(),
    Linear(in_features=4, out_features=4, bias=True),
)	
kwargs=dict(
)
INFO:trace: Series.__init__: ENTERING	
args=tuple(
    <class '__main__.ResNet'>,
    Linear(in_features=4, out_features=4, bias=True),
    ReLU(),
    Linear(in_features=4, out_features=4, bias=True),
)	
kwargs=dict(
)
INFO:trace: Series.__init__: SUCCESS with result=None
INFO:trace: Series.__init__: EXITING
INFO:trace: ResNet.__init__: SUCCESS with result=None
INFO:trace: ResNet.__init__: EXITING
INFO:trace: autojit.<locals>.WrappedClass.__new__: SUCCESS with result=ResNet(
  (0): Linear(in_features=4, out_features=4, bias=True)
 

cls=<class '__main__.ResNet'>, args=(Linear(in_features=4, out_features=4, bias=True), ReLU(), Linear(in_features=4, out_features=4, bias=True)), kwargs={}
__INIT__ CALLED
__INIT__ CALLED


ResNet(
  (0): Linear(in_features=4, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=4, bias=True)
)

In [ ]:
y = model(x)
torch.linalg.norm(y).backward()

In [15]:
class A:
    @trace
    def __new__(cls, *args, **kwargs):
        return  super().__new__(cls, *args, **kwargs)
    
    @trace
    def __init__(self):
        super().__init__()
    
class B:
    @trace
    def __new__(cls, *args, **kwargs):
        return  super().__new__(cls, *args, **kwargs)
    
    @trace
    def __init__(self):
        super().__init__()
    
    
obj = A()  

INFO:trace: A.__new__: ENTERING	
args=tuple(
    <class '__main__.A'>,
)	
kwargs=dict(
)
INFO:trace: A.__new__: SUCCESS with result=<__main__.A object at 0x7ffa027e2760>
INFO:trace: A.__new__: EXITING
INFO:trace: A.__init__: ENTERING	
args=tuple(
)	
kwargs=dict(
)
INFO:trace: A.__init__: SUCCESS with result=None
INFO:trace: A.__init__: EXITING


In [16]:
obj=B()

INFO:trace: B.__new__: ENTERING	
args=tuple(
    <class '__main__.B'>,
)	
kwargs=dict(
)
INFO:trace: B.__new__: SUCCESS with result=<__main__.B object at 0x7ffa027c89a0>
INFO:trace: B.__new__: EXITING
INFO:trace: B.__init__: ENTERING	
args=tuple(
)	
kwargs=dict(
)
INFO:trace: B.__init__: SUCCESS with result=None
INFO:trace: B.__init__: EXITING


In [28]:
class MyModule(nn.Module):
    
    
    def __init__(self) -> None:
        super().__init__()

        self.layer = nn.Identity()
        

In [29]:
MyModule()

MyModule(
  (layer): Identity()
)